# Integrator regression vs JPL Horizons

Pass a real asteroid's osculating elements (from Horizons) into the production n-body
integrator `propagate_elements_nbody` (`src/velocity_density_pipeline_gmm.py`), propagate,
and compare the result to Horizons at the target time — both the **state** (position) and
the recovered **orbital elements**.

Example object: **2024 YR4**. Change `OBJECT` / `EPOCH_MJD` / `TARGET_MJD` to test others.

**Runs on Hyak** (needs `assist` + `rebound` + `sorcha` + the ASSIST kernel). The Horizons
cells run anywhere `astroquery` has network; the propagation cell is the only Hyak-only one.

In [1]:
import sys, numpy as np
sys.path.insert(0, "../../src")   # for velocity_density_pipeline_gmm
from astroquery.jplhorizons import Horizons
from astropy.time import Time

OBJECT     = "2024 YR4"
EPOCH_MJD  = 60600.0     # TDB: epoch at which we take the elements (start of propagation)
TARGET_MJD = 61642.0     # TDB: propagate to here, then compare to Horizons
GR_MODEL   = "GR_SIMPLE" # 'GR_SIMPLE' = Sorcha parity; 'GR_EIH' = ASSIST full model

AU_KM   = 149597870.7
DAY_S   = 86400.0
GM_SUN  = 1.32712440018e11   # km^3/s^2 (for the state->elements check only)
print(f"{OBJECT}: propagate elements @ MJD {EPOCH_MJD} (TDB) -> MJD {TARGET_MJD} (TDB), span {(TARGET_MJD-EPOCH_MJD)/365.25:.2f} yr")

2024 YR4: propagate elements @ MJD 60600.0 (TDB) -> MJD 61642.0 (TDB), span 2.85 yr


## 1. Elements from Horizons at the start epoch

In [2]:
def horizons_elements(obj, mjd_tdb):
    """Heliocentric ecliptic J2000 osculating elements at mjd_tdb."""
    q = Horizons(id=obj, location='@sun', epochs=2400000.5 + mjd_tdb, id_type='smallbody')
    el = q.elements(refsystem='J2000', refplane='ecliptic')[0]
    return dict(a=float(el['a']), e=float(el['e']), incl=float(el['incl']),
                Omega=float(el['Omega']), w=float(el['w']),
                Tp_mjd=float(el['Tp_jd']) - 2400000.5, epoch_mjd=float(el['datetime_jd']) - 2400000.5)

el0 = horizons_elements(OBJECT, EPOCH_MJD)
for k, v in el0.items():
    print(f"  {k:9s} = {v}")

  a         = 2.539084380929908
  e         = 0.6641390888394662
  incl      = 3.452781847706646
  Omega     = 271.412339198305
  w         = 134.6422411098297
  Tp_mjd    = 60636.6354464991
  epoch_mjd = 60600.0


## 2. Propagate through the integrator  (Hyak-only)

In [3]:
import velocity_density_pipeline_gmm as vdp

r_km, v_kms = vdp.propagate_elements_nbody(
    a_AU=[el0['a']], e=[el0['e']], inc_deg=[el0['incl']],
    raan_deg=[el0['Omega']], argp_deg=[el0['w']],
    tp_mjd=[el0['Tp_mjd']], epoch_mjd=[el0['epoch_mjd']],
    obstime_str=Time(TARGET_MJD, format='mjd', scale='tdb'),
    show_progress=False, gr_model=GR_MODEL,
)
r_int = r_km[0]                    # heliocentric ecliptic, km
v_int = v_kms[0]                   # km/s
print(f"integrator state @ MJD {TARGET_MJD} (heliocentric ecliptic):")
print(f"  r = {r_int} km   |r| = {np.linalg.norm(r_int)/AU_KM:.6f} AU")
print(f"  v = {v_int} km/s")

/astro/users/ds2004/.conda/envs/neofast_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


integrator state @ MJD 61642.0 (heliocentric ecliptic):
  r = [-2.29299073e+08 -5.09975432e+08 -1.43784987e+07] km   |r| = 3.738948 AU
  v = [10.9526308   1.2712449   0.65372597] km/s


## 3. Horizons truth at the target time, and position comparison

In [4]:
def horizons_state(obj, mjd_tdb):
    """Heliocentric ecliptic state (km, km/s) at mjd_tdb."""
    q = Horizons(id=obj, location='@sun', epochs=2400000.5 + mjd_tdb, id_type='smallbody')
    v = q.vectors(refplane='ecliptic')[0]
    r = np.array([float(v['x']), float(v['y']), float(v['z'])]) * AU_KM
    vv = np.array([float(v['vx']), float(v['vy']), float(v['vz'])]) * AU_KM / DAY_S
    return r, vv

r_hz, v_hz = horizons_state(OBJECT, TARGET_MJD)
dr = np.linalg.norm(r_int - r_hz)
dv = np.linalg.norm(v_int - v_hz)
delta_au = np.linalg.norm(r_hz) 
ang_arcsec = np.degrees(dr / np.linalg.norm(r_hz)) * 3600   # heliocentric angular, rough
print(f"Horizons state @ MJD {TARGET_MJD}:")
print(f"  r = {r_hz} km   |r| = {np.linalg.norm(r_hz)/AU_KM:.6f} AU")
print()
print(f"|dr| integrator - Horizons = {dr:.3f} km  ({dr/AU_KM:.3e} AU)")
print(f"|dv|                       = {dv*1000:.4f} m/s")
print(f"span = {(TARGET_MJD-EPOCH_MJD)/365.25:.2f} yr  ->  {dr/((TARGET_MJD-EPOCH_MJD)/365.25):.2f} km/yr")

Horizons state @ MJD 61642.0:
  r = [-2.29299073e+08 -5.09975432e+08 -1.43784987e+07] km   |r| = 3.738948 AU

|dr| integrator - Horizons = 0.140 km  (9.391e-10 AU)
|dv|                       = 0.0000 m/s
span = 2.85 yr  ->  0.05 km/yr


## 4. Recovered elements at the target time vs Horizons

In [5]:
def rv_to_elements(r, v, mu=GM_SUN):
    """Classical osculating elements from heliocentric state (km, km/s)."""
    R = np.linalg.norm(r); V = np.linalg.norm(v)
    h = np.cross(r, v); H = np.linalg.norm(h)
    n = np.cross([0,0,1.0], h); N = np.linalg.norm(n)
    evec = (np.cross(v, h)/mu) - r/R; e = np.linalg.norm(evec)
    a = 1.0 / (2.0/R - V*V/mu)
    i = np.degrees(np.arccos(h[2]/H))
    Om = np.degrees(np.arctan2(n[1], n[0])) % 360
    w  = np.degrees(np.arccos(np.clip(np.dot(n, evec)/(N*e), -1, 1)))
    if evec[2] < 0: w = 360 - w
    return dict(a=a/AU_KM, e=e, incl=i, Omega=Om % 360, w=w % 360)

el_int = rv_to_elements(r_int, v_int)
el_hz  = horizons_elements(OBJECT, TARGET_MJD)

import pandas as pd
rows = []
for k, unit in [('a','AU'), ('e',''), ('incl','deg'), ('Omega','deg'), ('w','deg')]:
    iv, hv = el_int[k], el_hz[k]
    rows.append({'element': f'{k} ({unit})'.strip(), 'integrator': round(iv,8),
                 'Horizons': round(hv,8), 'diff': f'{iv-hv:+.2e}'})
display(pd.DataFrame(rows).set_index('element'))

,integrator,Horizons,diff
element,,,
a (AU),2.516462,2.516462,+5.78e-11
e (),0.661078,0.661078,-9.20e-11
incl (deg),3.407233,3.407233,-8.82e-10
Omega (deg),271.378668,271.378668,+6.79e-09
w (deg),134.342595,134.342595,-9.05e-09


## 5. Drift sweep — how the error grows with propagation span

Same as the Apophis check in `fixing_integrator.md` §9.9C: propagate the fixed start-epoch
elements to a range of target times and report |dr| vs Horizons. Pre-encounter this should
stay ~km/yr; a deep planetary encounter amplifies it (chaos, not a bug).

In [6]:
offsets_yr = [0.25, 0.5, 1.0, 2.0, 3.0]
rows = []
for dy in offsets_yr:
    tgt = EPOCH_MJD + dy * 365.25
    rk, _ = vdp.propagate_elements_nbody(
        a_AU=[el0['a']], e=[el0['e']], inc_deg=[el0['incl']], raan_deg=[el0['Omega']],
        argp_deg=[el0['w']], tp_mjd=[el0['Tp_mjd']], epoch_mjd=[el0['epoch_mjd']],
        obstime_str=Time(tgt, format='mjd', scale='tdb'), show_progress=False, gr_model=GR_MODEL)
    rh, _ = horizons_state(OBJECT, tgt)
    d = np.linalg.norm(rk[0] - rh)
    rows.append({'span_yr': dy, 'target_MJD': round(tgt,1), '|dr|_km': round(d,3), 'km/yr': round(d/dy,3)})
display(pd.DataFrame(rows).set_index('span_yr'))

,target_MJD,|dr|_km,km/yr
span_yr,,,
0.25,60691.3,0.003,0.014
0.50,60782.6,0.013,0.026
1.00,60965.2,0.036,0.036
2.00,61330.5,0.086,0.043
3.00,61695.8,0.153,0.051
